# 11 — Temporal Feature Engineering for Next-Month Dengue Forecasting

**Project:** Dengue Disease Prediction — Sri Lanka  
**Input:** The validated monthly integration files produced by Notebook 09  
**Output:** Reproducible, district-wise temporal features for **one-month-ahead dengue case prediction**

### Research rules

- One observation is one **administrative district in one calendar month**.
- For a feature month **t**, predict dengue cases in **t + 1 month**. Current-month dengue and weather are available at the end of month *t*; the target is **not** available.
- Create lags and rolling windows **within each district**, in chronological order. Never borrow values from another district or use future values.
- Use only months with complete weather coverage for all required historical weather windows. January 2010 and December 2024 are flagged as incomplete by Notebook 05/09.
- Population starts in **2014**. Keep a weather-only variant from 2010 and a separate population-inclusive variant from 2014. Do not invent 2010–2013 population.
- **2024 is reserved for final forecasting testing.** This notebook only builds deterministic features; it does not select features using test performance, scale variables, or fit a model.
- Notebook 12 will split by **target date**, not by the feature date. A December 2023 feature row predicting January 2024 belongs to the **2024 test set**.

> **Caution:** Monthly dengue values are proportional allocations from weekly reports and can be fractional. They are suitable for the current analysis but should not be described as independently observed integer monthly case counts.

## Step 1 — Import libraries and locate your project

Run from `D:\Dengue_Project\notebooks` in VS Code (or the project root). The code searches the current directory and its parents.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Open D:\\Dengue_Project in VS Code, "
        "then run this notebook from notebooks/ or the project root."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FINAL_DIR = PROJECT_ROOT / "data" / "final"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

KEYS = ["administrative_district", "date"]
WEATHER_VARS = [
    "temperature_mean_c", "humidity_mean_percent",
    "rainfall_total_mm", "wind_speed_mean_kmh",
]
TARGET_COL = "target_dengue_cases_t_plus_1"
print("Project root:", PROJECT_ROOT)
print("Forecast horizon: one calendar month")

Project root: d:\Dengue_Project
Forecast horizon: one calendar month


## Step 2 — Load the two integrated datasets from Notebook 09

The full dengue–weather file contains 2010–2024; the population-inclusive file contains 2014–2024. We compute temporal history from the **full dengue–weather file first** and add population afterward. This avoids discarding the valid 2013 history needed for features in early 2014.

In [2]:
files = {
    "dengue_weather": PROCESSED_DIR / "dengue_weather_monthly_2010_2024.csv",
    "population": PROCESSED_DIR / "dengue_weather_population_monthly_2014_2024.csv",
}
for label, path in files.items():
    print(f"{label:17s}: {path} | exists={path.is_file()}")
    if not path.is_file():
        raise FileNotFoundError(f"Run Notebook 09 first. Missing: {path}")

monthly = pd.read_csv(files["dengue_weather"], parse_dates=["date"])
monthly_population = pd.read_csv(files["population"], parse_dates=["date"])
print("Dengue-weather rows:", len(monthly))
print("Population-inclusive rows:", len(monthly_population))
display(monthly.head(3))

dengue_weather   : d:\Dengue_Project\data\processed\dengue_weather_monthly_2010_2024.csv | exists=True
population       : d:\Dengue_Project\data\processed\dengue_weather_population_monthly_2014_2024.csv | exists=True
Dengue-weather rows: 4500
Population-inclusive rows: 3300


,date,administrative_district,dengue_cases,temperature_mean_c,temperature_min_c,temperature_max_c,humidity_mean_percent,humidity_min_percent,humidity_max_percent,rainfall_total_mm,rainfall_max_hourly_mm,rainy_days,wind_speed_mean_kmh,wind_speed_max_kmh,complete_weather_month
0,2010-01-01,Ampara,116.428571,25.599833,22.20,28.95,80.270667,59.5,96.5,88.371429,4.35,21.714286,12.701000,24.15,False
1,2010-02-01,Ampara,124.428571,26.125893,22.20,31.40,78.548750,56.0,94.0,72.071429,4.35,12.571429,11.530714,24.15,True
2,2010-03-01,Ampara,72.857143,27.752097,22.55,32.55,75.846774,51.0,94.0,25.885714,2.45,14.500000,9.655000,21.65,True


## Step 3 — Verify keys, chronological continuity, and values

A one-row shift means "previous month" only when there are no missing district-month records. We therefore verify every district has a continuous monthly timeline **before** calculating lags.

In [3]:
def validate_monthly(frame, label, expected_rows, expected_months):
    required = set(KEYS + ["dengue_cases"] + WEATHER_VARS)
    if label == "population":
        required.add("population")
    else:
        required.add("complete_weather_month")
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{label} missing columns: {sorted(missing)}")
    if frame[KEYS].isna().any().any() or frame.duplicated(KEYS).any():
        raise ValueError(f"{label}: null or duplicate district-month keys")
    if not frame["date"].dt.is_month_start.all():
        raise ValueError(f"{label}: dates must be first-of-month")
    if frame["dengue_cases"].isna().any() or frame["dengue_cases"].lt(0).any():
        raise ValueError(f"{label}: missing or negative dengue cases")
    if frame[WEATHER_VARS].isna().any().any():
        raise ValueError(f"{label}: weather measurements contain NaN")
    if len(frame) != expected_rows or frame["administrative_district"].nunique() != 25:
        raise ValueError(f"{label}: unexpected row count or district count")
    ordered = frame.sort_values(KEYS).copy()
    month_index = ordered["date"].dt.year * 12 + ordered["date"].dt.month
    month_step = month_index.groupby(ordered["administrative_district"]).diff()
    if not month_step.dropna().eq(1).all():
        raise ValueError(f"{label}: missing months within a district")
    lengths = ordered.groupby("administrative_district").size()
    if not lengths.eq(expected_months).all():
        raise ValueError(f"{label}: not every district has {expected_months} months")
    print(f"PASS: {label}, {len(frame):,} rows, 25 districts, {expected_months} consecutive months")
    return ordered.reset_index(drop=True)

monthly = validate_monthly(monthly, "dengue_weather", 4500, 180)
monthly_population = validate_monthly(monthly_population, "population", 3300, 132)
assert monthly_population["population"].notna().all()
assert monthly_population["population"].gt(0).all()

PASS: dengue_weather, 4,500 rows, 25 districts, 180 consecutive months
PASS: population, 3,300 rows, 25 districts, 132 consecutive months


## Step 4 — Interpret weather completeness correctly

The weather completeness flag indicates whether the calendar month was fully covered by the original weekly data. It is **not** an imputation flag. Set incomplete weather measurements to `NaN` in a temporary working copy, so lag/rolling features cannot accidentally include them. Do **not** overwrite Notebook 09's data.

In [4]:
flag = monthly["complete_weather_month"].astype("string").str.strip().str.lower()
flag = flag.map({"true": True, "false": False, "1": True, "0": False})
if flag.isna().any():
    raise ValueError("Unrecognized complete_weather_month flag")
monthly["complete_weather_month"] = flag.astype(bool)

partial = monthly.loc[~monthly["complete_weather_month"], KEYS]
print("Incomplete-weather rows:", len(partial))
print("Incomplete months:", sorted(partial["date"].dt.strftime("%Y-%m").unique()))

working = monthly.copy()
working.loc[~working["complete_weather_month"], WEATHER_VARS] = np.nan
assert len(working) == 4500

Incomplete-weather rows: 50
Incomplete months: ['2010-01', '2024-12']


## Step 5 — Define the forecasting timeline

For each district:

| Available at end of feature month *t* | Prediction target |
|---|---|
| Dengue cases at *t*; weather at *t*; earlier dengue/weather; calendar month | Dengue cases at *t+1* |

Example: December 2023 measurements → January 2024 dengue cases. This example must be assigned to the **2024 test period**, even though its input month is in 2023.

We include current-month dengue as an input because the experiment assumes the month *t* report is available before making the prediction. If that assumption does not match your operational use case, revise the forecast origin before training.

In [5]:
features = working.copy()
by_district = features.groupby("administrative_district", sort=False)
features["target_date"] = features["date"] + pd.offsets.MonthBegin(1)
features[TARGET_COL] = by_district["dengue_cases"].shift(-1)

features["feature_year"] = features["date"].dt.year
features["feature_month_number"] = features["date"].dt.month
features["target_year"] = features["target_date"].dt.year
features["target_month_number"] = features["target_date"].dt.month

print(features.loc[features["administrative_district"] == "Colombo",
                   ["date", "dengue_cases", "target_date", TARGET_COL]].head(8))

          date  dengue_cases target_date  target_dengue_cases_t_plus_1
720 2010-01-01    561.142857  2010-02-01                    437.142857
721 2010-02-01    437.142857  2010-03-01                    250.142857
722 2010-03-01    250.142857  2010-04-01                    126.571429
723 2010-04-01    126.571429  2010-05-01                    313.142857
724 2010-05-01    313.142857  2010-06-01                    713.857143
725 2010-06-01    713.857143  2010-07-01                   1091.857143
726 2010-07-01   1091.857143  2010-08-01                    829.571429
727 2010-08-01    829.571429  2010-09-01                    238.571429


## Step 6 — Dengue lag features

`dengue_cases_lag1` = cases in month *t−1*. The current month's dengue count `dengue_cases` is also retained as a predictor. We add lags of 1, 2, 3 and 6 months, independently for each district.

In [6]:
DENGUE_LAGS = (1, 2, 3, 6)
for lag in DENGUE_LAGS:
    features[f"dengue_cases_lag{lag}"] = (
        features.groupby("administrative_district")["dengue_cases"].shift(lag)
    )

show = ["administrative_district", "date", "dengue_cases"] + [
    f"dengue_cases_lag{lag}" for lag in DENGUE_LAGS
]
display(features.loc[features["administrative_district"] == "Colombo", show].head(10))

,administrative_district,date,dengue_cases,dengue_cases_lag1,dengue_cases_lag2,dengue_cases_lag3,dengue_cases_lag6
720,Colombo,2010-01-01,561.142857,NaN,NaN,NaN,NaN
721,Colombo,2010-02-01,437.142857,561.142857,NaN,NaN,NaN
722,Colombo,2010-03-01,250.142857,437.142857,561.142857,NaN,NaN
723,Colombo,2010-04-01,126.571429,250.142857,437.142857,561.142857,NaN
724,Colombo,2010-05-01,313.142857,126.571429,250.142857,437.142857,NaN
725,Colombo,2010-06-01,713.857143,313.142857,126.571429,250.142857,NaN
726,Colombo,2010-07-01,1091.857143,713.857143,313.142857,126.571429,561.142857
727,Colombo,2010-08-01,829.571429,1091.857143,713.857143,313.142857,437.142857
728,Colombo,2010-09-01,238.571429,829.571429,1091.857143,713.857143,250.142857
729,Colombo,2010-10-01,199.285714,238.571429,829.571429,1091.857143,126.571429


## Step 7 — Weather lag features

Lagged weather may reflect delayed mosquito population and transmission responses. Add 1-, 2-, and 3-month weather lags for temperature, humidity, rainfall, and wind. These are **candidate features**, not claims of causation or a proven optimal lag.

In [7]:
WEATHER_LAGS = (1, 2, 3)
for variable in WEATHER_VARS:
    for lag in WEATHER_LAGS:
        features[f"{variable}_lag{lag}"] = (
            features.groupby("administrative_district")[variable].shift(lag)
        )
print("Created", len(WEATHER_VARS) * len(WEATHER_LAGS), "weather lag columns")

Created 12 weather lag columns


## Step 8 — Trailing rolling-window statistics

A trailing 3-month average at *t* uses **t−2, t−1, t**. A trailing 6-month average uses **t−5 through t**. These are known at the end of month *t*; **no centered windows and no future months** are used.

- Dengue: trailing 3- and 6-month mean.
- Temperature, humidity, wind: trailing 3- and 6-month mean.
- Rainfall: trailing 3- and 6-month **sum** (because monthly rainfall is already a total).

Use `min_periods=window` so any incomplete weather month inside a window makes that feature unavailable rather than silently using fewer months.

In [8]:
ROLL_WINDOWS = (3, 6)
for window in ROLL_WINDOWS:
    features[f"dengue_cases_roll{window}_mean"] = (
        features.groupby("administrative_district")["dengue_cases"]
        .transform(lambda s: s.rolling(window, min_periods=window).mean())
    )
    for variable in WEATHER_VARS:
        aggregation = "sum" if variable == "rainfall_total_mm" else "mean"
        name = f"{variable}_roll{window}_{aggregation}"
        features[name] = (
            features.groupby("administrative_district")[variable]
            .transform(
                lambda s, w=window, agg=aggregation:
                getattr(s.rolling(w, min_periods=w), agg)()
            )
        )

rolling_columns = [c for c in features.columns if "_roll" in c]
print("Rolling features:", rolling_columns)

Rolling features: ['dengue_cases_roll3_mean', 'temperature_mean_c_roll3_mean', 'humidity_mean_percent_roll3_mean', 'rainfall_total_mm_roll3_sum', 'wind_speed_mean_kmh_roll3_mean', 'dengue_cases_roll6_mean', 'temperature_mean_c_roll6_mean', 'humidity_mean_percent_roll6_mean', 'rainfall_total_mm_roll6_sum', 'wind_speed_mean_kmh_roll6_mean']


## Step 9 — Calendar-seasonality features

A known calendar month is not a leaked future observation. Sine and cosine encode the cyclic relationship between December and January. We encode the **target month** because it is known at prediction time.

\[
\mathrm{month\_sin} = \sin(2\pi m / 12),\qquad
\mathrm{month\_cos} = \cos(2\pi m / 12)
\]

In [9]:
features["target_month_sin"] = np.sin(
    2 * np.pi * features["target_month_number"] / 12
)
features["target_month_cos"] = np.cos(
    2 * np.pi * features["target_month_number"] / 12
)
assert features["target_month_sin"].between(-1, 1).all()
assert features["target_month_cos"].between(-1, 1).all()

## Step 10 — Define the predictor schema explicitly

This whitelist is important: **the target and any future dengue count must never be a predictor**. District name, dates, and year are retained as metadata for grouping and splitting, but not treated as numerical model inputs automatically.

The current weather and dengue predictors are measured at *t*. Population is added separately at *t* for the population-inclusive variant.

In [10]:
DENGUE_FEATURES = ["dengue_cases"] + [
    f"dengue_cases_lag{lag}" for lag in DENGUE_LAGS
] + [f"dengue_cases_roll{w}_mean" for w in ROLL_WINDOWS]

WEATHER_FEATURES = list(WEATHER_VARS) + [
    f"{var}_lag{lag}" for var in WEATHER_VARS for lag in WEATHER_LAGS
] + [
    f"{var}_roll{w}_{'sum' if var == 'rainfall_total_mm' else 'mean'}"
    for w in ROLL_WINDOWS for var in WEATHER_VARS
]

SEASONAL_FEATURES = ["target_month_sin", "target_month_cos"]
MODEL_FEATURES_NO_POPULATION = DENGUE_FEATURES + WEATHER_FEATURES + SEASONAL_FEATURES
MODEL_FEATURES_WITH_POPULATION = MODEL_FEATURES_NO_POPULATION + ["population"]

assert len(MODEL_FEATURES_NO_POPULATION) == len(set(MODEL_FEATURES_NO_POPULATION))
assert TARGET_COL not in MODEL_FEATURES_NO_POPULATION
assert "target_date" not in MODEL_FEATURES_NO_POPULATION
assert all(col in features.columns for col in MODEL_FEATURES_NO_POPULATION)
print("Predictor count without population:", len(MODEL_FEATURES_NO_POPULATION))
print("Predictor count with population:", len(MODEL_FEATURES_WITH_POPULATION))
print("Predictors:", MODEL_FEATURES_NO_POPULATION)

Predictor count without population: 33
Predictor count with population: 34
Predictors: ['dengue_cases', 'dengue_cases_lag1', 'dengue_cases_lag2', 'dengue_cases_lag3', 'dengue_cases_lag6', 'dengue_cases_roll3_mean', 'dengue_cases_roll6_mean', 'temperature_mean_c', 'humidity_mean_percent', 'rainfall_total_mm', 'wind_speed_mean_kmh', 'temperature_mean_c_lag1', 'temperature_mean_c_lag2', 'temperature_mean_c_lag3', 'humidity_mean_percent_lag1', 'humidity_mean_percent_lag2', 'humidity_mean_percent_lag3', 'rainfall_total_mm_lag1', 'rainfall_total_mm_lag2', 'rainfall_total_mm_lag3', 'wind_speed_mean_kmh_lag1', 'wind_speed_mean_kmh_lag2', 'wind_speed_mean_kmh_lag3', 'temperature_mean_c_roll3_mean', 'humidity_mean_percent_roll3_mean', 'rainfall_total_mm_roll3_sum', 'wind_speed_mean_kmh_roll3_mean', 'temperature_mean_c_roll6_mean', 'humidity_mean_percent_roll6_mean', 'rainfall_total_mm_roll6_sum', 'wind_speed_mean_kmh_roll6_mean', 'target_month_sin', 'target_month_cos']


## Step 11 — Drop only rows without a complete historical window or next-month target

Missing early lags are **expected**: a 6-month window cannot be calculated at the beginning of the series. Keep a clear audit of exclusions rather than filling those values with zeros. This filter is applied **after** features are calculated, so a future month cannot affect earlier rolling statistics.

In [11]:
features["has_next_month_target"] = features[TARGET_COL].notna()
features["has_all_predictors"] = features[MODEL_FEATURES_NO_POPULATION].notna().all(axis=1)
features["eligible_for_model"] = (
    features["has_next_month_target"] & features["has_all_predictors"]
)

feature_audit = (
    features.groupby("feature_year", as_index=False)
    .agg(
        input_rows=("date", "size"),
        complete_predictor_rows=("has_all_predictors", "sum"),
        next_month_target_rows=("has_next_month_target", "sum"),
        eligible_rows=("eligible_for_model", "sum"),
    )
)
display(feature_audit)

META = [
    "administrative_district", "date", "target_date", "feature_year",
    "feature_month_number", "target_year", "target_month_number",
]

weather_only = (
    features.loc[features["eligible_for_model"],
                 META + MODEL_FEATURES_NO_POPULATION + [TARGET_COL]]
    .sort_values(KEYS).reset_index(drop=True)
)
print("Weather-only feature dataset:", weather_only.shape)
print("Feature months:", weather_only["date"].min(), "to", weather_only["date"].max())
print("Target months:", weather_only["target_date"].min(), "to", weather_only["target_date"].max())

,feature_year,input_rows,complete_predictor_rows,next_month_target_rows,eligible_rows
0,2010,300,150,300,150
1,2011,300,300,300,300
2,2012,300,300,300,300
3,2013,300,300,300,300
4,2014,300,300,300,300
5,2015,300,300,300,300
6,2016,300,300,300,300
7,2017,300,300,300,300
8,2018,300,300,300,300
9,2019,300,300,300,300


Weather-only feature dataset: (4325, 41)
Feature months: 2010-07-01 00:00:00 to 2024-11-01 00:00:00
Target months: 2010-08-01 00:00:00 to 2024-12-01 00:00:00


## Step 12 — Join population at the **feature month**, not the target month

Population at time *t* is an input to predict dengue at *t+1*. Never merge the population of *t+1* into the predictor row. The original 2010–2013 structural gap remains absent from this variant.

In [12]:
pop_keys = monthly_population[KEYS + ["population"]].copy()
with_population = weather_only.merge(
    pop_keys,
    on=KEYS,
    how="inner",
    validate="one_to_one",
)
with_population = with_population[
    META + MODEL_FEATURES_WITH_POPULATION + [TARGET_COL]
].sort_values(KEYS).reset_index(drop=True)

assert with_population["date"].min() == pd.Timestamp("2014-01-01")
assert with_population["population"].notna().all()
assert with_population["population"].gt(0).all()
assert with_population["target_date"].dt.year.max() == 2024
print("Population-inclusive feature dataset:", with_population.shape)
print("Feature months:", with_population["date"].min(), "to", with_population["date"].max())
print("Target months:", with_population["target_date"].min(), "to", with_population["target_date"].max())

Population-inclusive feature dataset: (3275, 42)
Feature months: 2014-01-01 00:00:00 to 2024-11-01 00:00:00
Target months: 2014-02-01 00:00:00 to 2024-12-01 00:00:00


## Step 13 — Validate target alignment and absence of future leakage

We test the engineered columns against the original integrated data for several districts and dates, including a year boundary. This is a **data-engineering check**, not a model evaluation.

In [13]:
lookup = monthly.set_index(KEYS).sort_index()
for frame, label, cols in [
    (weather_only, "weather-only", MODEL_FEATURES_NO_POPULATION),
    (with_population, "population-inclusive", MODEL_FEATURES_WITH_POPULATION),
]:
    if frame.duplicated(KEYS).any() or frame[KEYS].isna().any().any():
        raise AssertionError(f"{label}: duplicate or missing district-month key")
    if frame[cols + [TARGET_COL]].isna().any().any():
        raise AssertionError(f"{label}: incomplete model features or targets")
    if not (frame["target_date"] == frame["date"] + pd.offsets.MonthBegin(1)).all():
        raise AssertionError(f"{label}: wrong target horizon")
    if not (frame["target_date"] > frame["date"]).all():
        raise AssertionError(f"{label}: future target alignment failed")
    if not frame[TARGET_COL].ge(0).all():
        raise AssertionError(f"{label}: negative target")
    for row in frame.iloc[::max(1, len(frame)//20)].itertuples(index=False):
        expected = lookup.loc[(row.administrative_district, row.target_date), "dengue_cases"]
        if not np.isclose(getattr(row, TARGET_COL), expected):
            raise AssertionError(f"{label}: target mismatch at {row.administrative_district}, {row.date}")
    print(f"PASS: {label} — keys, dates, non-null predictors, and next-month targets")

# Spot-check a known 2024 test boundary, without using its target to select features.
jan_test = weather_only.loc[weather_only["target_date"].eq(pd.Timestamp("2024-01-01"))]
assert len(jan_test) == 25
assert jan_test["date"].eq(pd.Timestamp("2023-12-01")).all()
print("PASS: December 2023 inputs -> January 2024 test targets for 25 districts")

PASS: weather-only — keys, dates, non-null predictors, and next-month targets
PASS: population-inclusive — keys, dates, non-null predictors, and next-month targets
PASS: December 2023 inputs -> January 2024 test targets for 25 districts


## Step 14 — Preview the chronological split (Notebook 12 will implement it)

The split is defined by **`target_date`**, because the objective is to forecast cases in the target month. These dates are research-design defaults and can be revised in Notebook 12 **before model tuning**, but 2024 must remain the untouched final test year.

- Train targets: through December 2021.
- Validation targets: January 2022–December 2023.
- Test targets: January 2024–December 2024.

Temporal feature calculations can use historical months from a previous partition, as long as all inputs precede their prediction targets. Scalers and learned imputers must be fit on the **training partition only** in Notebook 12.

In [14]:
def split_label(target_date):
    if target_date < pd.Timestamp("2022-01-01"):
        return "train"
    if target_date < pd.Timestamp("2024-01-01"):
        return "validation"
    return "test"

split_rows = []
for name, frame in [("weather_only", weather_only),
                    ("with_population", with_population)]:
    labels = frame["target_date"].map(split_label)
    for partition in ["train", "validation", "test"]:
        portion = frame.loc[labels.eq(partition)]
        split_rows.append({
            "dataset": name,
            "partition": partition,
            "rows": len(portion),
            "districts": portion["administrative_district"].nunique(),
            "first_target_date": portion["target_date"].min(),
            "last_target_date": portion["target_date"].max(),
        })

split_preview = pd.DataFrame(split_rows)
display(split_preview)
assert split_preview.loc[split_preview["partition"].eq("test"), "rows"].eq(25*12).all()
print("PASS: each variant contains 12 target months in 2024 for 25 districts")

,dataset,partition,rows,districts,first_target_date,last_target_date
0,weather_only,train,3425,25,2010-08-01,2021-12-01
1,weather_only,validation,600,25,2022-01-01,2023-12-01
2,weather_only,test,300,25,2024-01-01,2024-12-01
3,with_population,train,2375,25,2014-02-01,2021-12-01
4,with_population,validation,600,25,2022-01-01,2023-12-01
5,with_population,test,300,25,2024-01-01,2024-12-01


PASS: each variant contains 12 target months in 2024 for 25 districts


## Step 15 — Save the feature datasets and audit tables

`data/final/` contains **unscaled, unencoded** feature datasets. Do not fit a scaler or train an LSTM here. Notebook 12 will load these files and create time-based partitions and train-only transformations.

The schema file lists the exact predictor names to prevent accidental use of the target, date, or district identifier as a numerical input.

In [15]:
feature_dictionary = pd.DataFrame([
    {
        "feature": name,
        "category": (
            "population" if name == "population" else
            "seasonality" if name in SEASONAL_FEATURES else
            "dengue" if name in DENGUE_FEATURES else "weather"
        ),
        "availability": "end_of_feature_month_t",
        "description": (
            "Population for feature month t; 2014 onward only" if name == "population" else
            "Known calendar position of target month t+1" if name in SEASONAL_FEATURES else
            "Historical or current-month predictor; no future observations"
        ),
    }
    for name in MODEL_FEATURES_WITH_POPULATION
])

output_files = {
    FINAL_DIR / "dengue_temporal_features_weather_only_2010_2024.csv": weather_only,
    FINAL_DIR / "dengue_temporal_features_with_population_2014_2024.csv": with_population,
    TABLES_DIR / "notebook11_feature_availability_audit.csv": feature_audit,
    TABLES_DIR / "notebook11_split_preview.csv": split_preview,
    TABLES_DIR / "notebook11_feature_dictionary.csv": feature_dictionary,
}
for path, frame in output_files.items():
    frame.to_csv(path, index=False)
    print(f"Saved: {path.relative_to(PROJECT_ROOT)}  ({len(frame):,} rows)")

schema = {
    "task": "one_month_ahead_dengue_case_forecasting",
    "forecast_origin": "end_of_feature_month_t",
    "forecast_target": TARGET_COL,
    "target_date_column": "target_date",
    "feature_date_column": "date",
    "district_column": "administrative_district",
    "weather_only_predictors": MODEL_FEATURES_NO_POPULATION,
    "population_predictors": MODEL_FEATURES_WITH_POPULATION,
    "split_basis": "target_date",
    "train_target_end": "2021-12-01",
    "validation_target_end": "2023-12-01",
    "test_target_year": 2024,
    "scaling": "not_performed_train_only_in_notebook12",
}
schema_path = FINAL_DIR / "notebook11_temporal_feature_schema.json"
schema_path.write_text(json.dumps(schema, indent=2), encoding="utf-8")
print("Saved:", schema_path.relative_to(PROJECT_ROOT))

Saved: data\final\dengue_temporal_features_weather_only_2010_2024.csv  (4,325 rows)
Saved: data\final\dengue_temporal_features_with_population_2014_2024.csv  (3,275 rows)
Saved: results\tables\notebook11_feature_availability_audit.csv  (15 rows)
Saved: results\tables\notebook11_split_preview.csv  (6 rows)
Saved: results\tables\notebook11_feature_dictionary.csv  (34 rows)
Saved: data\final\notebook11_temporal_feature_schema.json


## Step 16 — Final audit and expected results

With the existing 25-district, 2010–2024 monthly integration, a six-month trailing weather window, and incomplete weather in January 2010 / December 2024:

| Dataset | Expected eligible feature months | Expected district-month rows | Forecast target coverage |
|---|---:|---:|---|
| Weather-only | July 2010–November 2024 | **4,325** | August 2010–December 2024 |
| With population | January 2014–November 2024 | **3,275** | February 2014–December 2024 |

These expected values are **validation checks**, not model accuracy results. If the counts differ, investigate the inputs or completeness flags rather than forcing a match.

In [16]:
assert len(weather_only) == 4325, (
    f"Expected 4,325 weather-only rows; got {len(weather_only)}. "
    "Review incomplete months and rolling-window logic."
)
assert len(with_population) == 3275, (
    f"Expected 3,275 population-inclusive rows; got {len(with_population)}"
)
assert weather_only["date"].min() == pd.Timestamp("2010-07-01")
assert weather_only["date"].max() == pd.Timestamp("2024-11-01")
assert with_population["date"].min() == pd.Timestamp("2014-01-01")
assert with_population["date"].max() == pd.Timestamp("2024-11-01")
assert weather_only["target_date"].max() == pd.Timestamp("2024-12-01")
assert with_population["target_date"].max() == pd.Timestamp("2024-12-01")
assert all(path.is_file() for path in output_files)
assert schema_path.is_file()
print("ALL NOTEBOOK 11 CHECKS PASSED")
print("Next: Notebook 12 — Time-Series Split and Train-Only Scaling")

ALL NOTEBOOK 11 CHECKS PASSED
Next: Notebook 12 — Time-Series Split and Train-Only Scaling


## Interpretation and research notes

1. **Feature construction is deterministic.** Lagging and rolling statistics do not involve fitting on test targets; nonetheless, do not inspect 2024 results to select features or hyperparameters.
2. **2024 is not a development set.** All exploratory choices and model tuning should use training/validation data. Final evaluation on 2024 is done once after choices are fixed.
3. **Historical weather completeness matters.** A row is removed if any required weather input within the six-month lookback is incomplete. The January 2010 boundary does not contaminate later rows.
4. **Population is a separate experiment.** The weather-only dataset retains earlier observations; the population-inclusive dataset has a shorter period. Compare models on the **same target-date test observations** and be cautious when comparing different training histories.
5. **Current-month dengue is an assumption.** This is an end-of-month-*t* prediction of month *t+1*. If operational reporting delays prevent the current month's dengue count from being known, the input timing must be redesigned.
6. **Monthly population data is annual data repeated by month.** It does not represent newly measured monthly population changes.
7. **No model is trained here.** LSTM sequence construction, train-only scaling, chronological splits, and evaluation belong in subsequent notebooks.

### Before pushing to GitHub

Run **Restart Kernel → Run All**, inspect the two final datasets and audit tables, save the executed notebook, and commit the notebook and intended output files. Check `git status` first to avoid staging unrelated or sensitive data.